In [1]:
import json

from dynamarq import *

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [2]:
service = QiskitRuntimeService()

# Retrieve all backends that support dynamic circuits
compatible_backends = service.backends(dynamic_circuits=True)

for backend in compatible_backends:
    print(backend.name)

qiskit_runtime_service.__init__:WARNING:2026-06-06 07:56:14,777: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (premium), the available account instances are: m5000-eu, m5000-us. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-06-06 07:56:14,778: Loading instance: m5000-eu, plan: premium
qiskit_runtime_service.backends:WARNING:2026-06-06 07:56:15,435: Loading instance: m5000-us, plan: premium


ibm_aachen
ibm_pittsburgh
ibm_marrakesh
ibm_kingston
ibm_boston
ibm_fez


In [15]:
backend_name = 'ibm_fez'
backend = service.backend(backend_name)
print(backend)

qiskit_runtime_service.backends:WARNING:2026-06-06 07:58:23,220: Using instance: m5000-us, plan: premium


<IBMBackend('ibm_fez')>


In [16]:
def submit_benchmark_dd(bm: benchmark.Benchmark) :
    name = bm.name()
    circuits = bm.qiskit_circuits(stretch_dd=True)

    pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
    isa_circuits = [pm.run(circuit) for circuit in circuits]

    jobs = []
    for isa_circuit in isa_circuits :
        sampler = Sampler(backend)
        job = sampler.run([isa_circuit])
        job_id = job.job_id()
        jobs.append(job_id)

    with open(f"{name}_dd_job.json", "r") as f :
        json.dump(jobs, f, indent=2)

In [17]:
rus = RepeatUntilSuccess.RepeatUntilSuccess(3)

In [18]:
submit_benchmark_dd(rus)

TranspilerError: "The control-flow construct 'while_loop' is not supported by the backend."

In [ ]:
def extract_dd_counts(bm: benchmark.Benchmark) :
    name = bm.name()

    with open(f"{name}_dd_job.json", "r") as f :
        job_ids = json.load(f)

    counts_list = []
    for job_id in job_ids :
        job = service.job(job_id)
        result = job.result()
        print(name, result[0].data)
        if 'FiveQubitCode' in name :
            counts = result[0].data.result.get_counts()
        elif 'IPE_3_2' in name :
            counts = result[0].data.c0.get_counts()
        elif 'IPE_5_3' in name :
            counts = result[0].data.c1.get_counts()
        elif 'IPE_21_5' in name :
            counts = result[0].data.c2.get_counts()
        elif 'IPE_682_10' in name :
            counts = result[0].data.c3.get_counts()
        elif 'CNOTLadder' in name or 'Fanout' in name or 'LongRangeCNOT' in name:
            counts = result[0].data.c.get_counts()
        elif 'RepeatUntilSuccess' in name :
            counts = result[0].data.c0.get_counts()
        else :
            counts = result[0].data.meas.get_counts()
        counts_list.append(counts)

    with open(f"{name}_dd_counts.json", "w") as f :
        json.dump(counts_list, f, indent=2)
    return counts_list